In [1]:
import pandas as pd
# 1. Importation des données
df= pd.read_csv("store.csv")

In [2]:
# 2. Identification des anomalies avec l'IQR

Q1, Q3 = df['Sales'].quantile([0.25, 0.75])
limit = Q3 + 1.5 * (Q3 - Q1)
anos = df[df['Sales'] > limit]
print(f"Nombre d'anomalies (> {limit:.2f} $) : {len(anos)}")
print("\nLes Catégories qui ont le plus d'anomalies :\n", anos['Category'].value_counts().head(3))

Nombre d'anomalies (> 500.64 $) : 1145

Les Catégories qui ont le plus d'anomalies :
 Category
Furniture          459
Technology         394
Office Supplies    292
Name: count, dtype: int64


In [3]:
# 3. Nettoyage des données

df.dropna(subset=['Postal Code'], inplace=True)
df.drop_duplicates(inplace=True)
print(df.columns)
df.info()

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales'],
      dtype='str')
<class 'pandas.DataFrame'>
Index: 9789 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9789 non-null   int64  
 1   Order ID       9789 non-null   str    
 2   Order Date     9789 non-null   str    
 3   Ship Date      9789 non-null   str    
 4   Ship Mode      9789 non-null   str    
 5   Customer ID    9789 non-null   str    
 6   Customer Name  9789 non-null   str    
 7   Segment        9789 non-null   str    
 8   Country        9789 non-null   str    
 9   City           9789 non-null   str    
 10  State          9789 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         

In [4]:
# 4. Formatage des dates
df['Order Date']= pd.to_datetime(df['Order Date'],format="%d/%m/%Y",errors='coerce')
df['Ship Date']= pd.to_datetime(df['Ship Date'],format="%d/%m/%Y",errors='coerce')
# 5. Extraction temporelle
df['years']=df['Order Date'].dt.year
df['quarter']=df['Order Date'].dt.quarter
df['months']=df['Order Date'].dt.month
# 6. Calculs financiers (Finance)
df['cost']=(df['Sales']*0.60).round(2)
df['profit']=(df['Sales']-df['cost']).round(2)
df['ratio_profit']=((df['profit'] / df['Sales'])*100).round(2)

In [5]:
# 7. Calcul du délai de livraison (Logistique)
df['delivery_days']=(df['Ship Date']-df['Order Date']).dt.days

In [6]:
# 8. Segmentation des clients (Commercial)
def segment_valeur(valeur):
    if valeur > 500:
        return 'high value'
    elif valeur > 250:
        return 'medium value'
    else: 
        return 'low value'
# 9. Agrégation des données (KPIs)
df['categorise']=df['Sales'].apply(segment_valeur)

In [7]:
rev_p_rj=df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
rev_p_prd=df.groupby(["Product ID","Product Name"])["Sales"].sum().sort_values(ascending=False)
rev_p_prd
rev_p_rj

Region
West       710219.6845
East       660589.3560
Central    492646.9132
South      389151.4590
Name: Sales, dtype: float64

In [8]:
# 10. Exportation finale
df.to_csv("store_new.csv",index=False)